# 09 (heavy) - SAMap run, HLiCA x mouse

sc-samap 3.0.1 (Project 23 WP2 recipe): SAMAP(filenames, f_maps), sm.run(), get_mapping_scores(n_top=0),
GenePairFinder(align_thr=0.1). Record: 62 min on 16 CPUs; outputs ~23 GB (pickled object 14 GB).

**Heavy notebook.** Nothing runs until you set `RUN_HEAVY = True` in the cell below. Long steps start in a detached
`screen` session with an exit sentinel (`WORK/logs/<name>.exit`); the wait cell polls the log. Re-run the light notebook named
below afterwards so that it uses these rebuilt outputs instead of the record's. Afterwards: notebook 08.

**Runtime:** ~1 h, 16 CPU, ~25 GB disk
**Envs used (each step runs as a subprocess in its own env):** samap
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
RUN_HEAVY = False   # set True to run

In [ ]:
if RUN_HEAVY:   # nothing is touched unless the notebook is really run (review fix)
    guard(40)
    link("samap_run/input", "samap_maps/liver_hsmm/maps_mmseqs")
    fresh("samap_run/out")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/02_samap.py
"""SAMap run HLiCA (hs) x Guilliams 2022 mouse (mm), sc-samap 3.0.1 (Project 23 WP2 recipe: SAMAP(filenames, f_maps), sm.run(),
get_mapping_scores n_top=0, GenePairFinder align_thr 0.1). Maps: our liver MMseqs2 maps (samap_maps/liver_hsmm/maps_mmseqs/hsmm).
Keys: hs author_cell_type (48 HLiCA labels), mm annotation_aor. Outputs /mnt/masld_vault/cross_species_repro/samap_run/out/."""
import os, pickle, time
import numpy as np, pandas as pd
from scipy import sparse
from samap import SAMAP
from samap.analysis import get_mapping_scores, GenePairFinder
from samap.utils import save_samap
t0 = time.time(); log = lambda s: print(f"[{(time.time() - t0) / 60:6.1f} min] {s}", flush=True)
I = "/mnt/masld_vault/cross_species_repro/samap_run/input"; O = "/mnt/masld_vault/cross_species_repro/samap_run/out"
MAPS = "/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm/maps_mmseqs/"
sm = SAMAP({"hs": f"{I}/hs_samap_ready.h5ad", "mm": f"{I}/mm_samap_ready.h5ad"}, f_maps=MAPS)
log("SAMAP object built")
sm.run(ncpus=16)
log("sm.run done")
keys = {"hs": "author_cell_type", "mm": "annotation_aor"}
D, MT = get_mapping_scores(sm, keys, n_top=0)
pd.DataFrame(D).to_csv(f"{O}/mapping_D.tsv", sep="\t"); MT.to_csv(f"{O}/mapping_table.tsv", sep="\t")
log("mapping scores written")
sm.samap.adata.write_h5ad(f"{O}/samap_joint.h5ad")
for sp_, sam in sm.sams.items():
    X = sam.adata.X
    if sparse.issparse(X):
        X = X.tocsr(copy=False); X.indices = X.indices.astype(np.int64, copy=False); X.indptr = X.indptr.astype(np.int64, copy=False); sam.adata.X = X
try:
    gpf = GenePairFinder(sm, keys=keys)
    gp = gpf.find_all(align_thr=0.1)
    pd.DataFrame(gp).to_csv(f"{O}/gene_pairs_thr0.1.tsv", sep="\t", index=False)
    log("gene pairs written")
except Exception as e:
    log(f"GenePairFinder failed: {e!r}")
save_samap(sm, f"{O}/samap_object.pkl")
log("done")

In [ ]:
launch("samap", [f"{PY['samap']} -I {WORK}/02_samap.py > {LOGS}/02_samap.py.log 2>&1"])
wait_for("samap", log=f"{LOGS}/02_samap.py.log")

In [ ]:
if RUN_HEAVY:
    import pandas as pd
    a = pd.read_csv(rec("samap_run/out/mapping_table.tsv"), sep="\t", index_col=0); b = pd.read_csv(rep("samap_run/out/mapping_table.tsv"), sep="\t", index_col=0)
    print("max |difference| of mapping scores vs record:", float((a - b.reindex_like(a)).abs().max().max()))
    # 02_samap.py logs a GenePairFinder failure instead of stopping; fail here if the gene pairs are missing (review fix)
    assert os.path.exists(rep("samap_run/out/gene_pairs_thr0.1.tsv")), "GenePairFinder failed: see the 02_samap log"